# Vehicle Specification Extraction — Notebook

This notebook contains the **single-file** pipeline split into runnable cells:

- PDF text extraction
- Chunking
- Embeddings (SentenceTransformers)
- FAISS retrieval
- Optional OpenAI LLM extraction

You can run cells one by one. Adjust parameters in the **Run** cell at the end.


In [ ]:

# Imports
import os, sys, re, json, getpass
from pathlib import Path
from typing import List, Dict, Any
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

try:
    import openai
    _OPENAI_AVAILABLE = True
except Exception:
    _OPENAI_AVAILABLE = False

print('Libraries loaded. OpenAI available:', _OPENAI_AVAILABLE)


In [ ]:

def extract_pdf_text(pdf_path: str) -> List[Dict[str, Any]]:
    """Extract clean text from each page of a PDF (PyMuPDF)."""
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(doc.page_count):
        page = doc.load_page(i)
        text = page.get_text("text")
        text = re.sub(r"\s+", " ", text).strip()
        pages.append({"page_num": i + 1, "text": text})
    return pages


In [ ]:

def chunk_texts(pages: List[Dict[str, Any]], chunk_size: int = 1000, overlap: int = 200) -> List[Dict[str, Any]]:
    """Split pages into overlapping text chunks."""
    chunks = []
    for p in pages:
        text = p.get('text', '')
        start = 0
        i = 0
        L = len(text)
        if L == 0:
            continue
        while start < L:
            end = min(start + chunk_size, L)
            chunk_text = text[start:end]
            chunks.append({
                'chunk_id': f"p{p['page_num']}_c{i}",
                'page': p['page_num'],
                'text': chunk_text,
                'start': start,
                'end': end
            })
            i += 1
            start = end - overlap
            if start < 0:
                start = 0
            if start >= L:
                break
    return chunks


In [ ]:

def build_embeddings(texts: List[str], model_name: str = 'all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    emb = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    emb = emb.astype('float32')
    return emb

def build_faiss_index(embeddings: np.ndarray):
    faiss.normalize_L2(embeddings)
    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    return index


In [ ]:

from dataclasses import dataclass

@dataclass
class Retriever:
    index: Any
    embeddings: np.ndarray
    chunks: List[Dict[str, Any]]
    emb_model_name: str = 'all-MiniLM-L6-v2'

    def __post_init__(self):
        self.model = SentenceTransformer(self.emb_model_name)

    def get_relevant_chunks(self, query: str, top_k: int = 5):
        q_emb = self.model.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        D, I = self.index.search(q_emb, top_k)
        results = []
        for idx, score in zip(I[0], D[0]):
            if idx < 0: continue
            chunk = self.chunks[int(idx)]
            results.append({'chunk_id': chunk['chunk_id'], 'page': chunk['page'], 'text': chunk['text'], 'score': float(score)})
        return results


In [ ]:

def build_prompt(query: str, chunks: List[Dict[str, Any]], max_chars: int = 8000) -> str:
    instruction = (
        "You are an assistant that extracts vehicle specifications from the provided text chunks. " 
        "Return a JSON array. Each element should have: component, spec_type, value, unit, source_page, chunk_id. "
        "If none, return []\n\n"
    )
    chunk_text = "\n\n---\n\n".join([f"Chunk {c['chunk_id']} (page {c['page']}):\n{c['text']}" for c in chunks])
    prompt = f"{instruction}Query: {query}\n\nText:\n{chunk_text}\n\nReturn JSON now."
    return prompt

def extract_specs_from_chunks_openai(query: str, chunks: List[Dict[str, Any]], openai_model: str = 'gpt-3.5-turbo') -> List[Dict[str, Any]]:
    if not _OPENAI_AVAILABLE:
        raise RuntimeError('OpenAI package not available. Install with pip install openai')
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY not set in environment.')
    openai.api_key = api_key
    prompt = build_prompt(query, chunks)
    resp = openai.ChatCompletion.create(model=openai_model, messages=[{'role':'user','content': prompt}], temperature=0)
    text = resp['choices'][0]['message']['content']
    m = re.search(r"(\[.*\])", text, re.S)
    if m:
        try:
            data = json.loads(m.group(1))
        except Exception:
            data = []
    else:
        data = []
    for item in data:
        if 'chunk_id' not in item and len(chunks)>0:
            item['chunk_id'] = chunks[0]['chunk_id']
        if 'source_page' not in item and len(chunks)>0:
            item['source_page'] = chunks[0]['page']
    return data


In [ ]:

def prompt_for_openai_key():
    key = getpass.getpass('Paste your OpenAI API key (hidden): ').strip()
    if not key:
        return False
    os.environ['OPENAI_API_KEY'] = key
    return True

def save_json(data, out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def save_csv(items, out_csv):
    if not items:
        print('Nothing to save to CSV (empty items).')
        return
    df = pd.DataFrame(items)
    df.to_csv(out_csv, index=False)


In [ ]:

def run_pipeline(pdf_path: str, query: str, use_openai: bool = False, top_k: int = 6, chunk_size: int = 1000, overlap: int = 200, emb_model: str = 'all-MiniLM-L6-v2', openai_model: str = 'gpt-3.5-turbo', out: str = 'extracted_specs.json', out_csv: str = None, verbose: bool = True):
    if use_openai and not os.getenv('OPENAI_API_KEY'):
        ok = prompt_for_openai_key()
        if not ok:
            print('OpenAI key not provided. Continuing without OpenAI.')
            use_openai = False

    if verbose: print('Extracting PDF text...')
    pages = extract_pdf_text(pdf_path)
    if verbose: print(f'Pages: {len(pages)}')

    if verbose: print('Chunking...')
    chunks = chunk_texts(pages, chunk_size=chunk_size, overlap=overlap)
    if verbose: print(f'Chunks: {len(chunks)}')

    if len(chunks) == 0:
        print('No chunks found. Exiting.')
        return []

    if verbose: print('Building embeddings... (this may take a while)')
    texts = [c['text'] for c in chunks]
    embeddings = build_embeddings(texts, model_name=emb_model)

    if verbose: print('Building FAISS index...')
    index = build_faiss_index(embeddings)
    retriever = Retriever(index=index, embeddings=embeddings, chunks=chunks, emb_model_name=emb_model)

    if verbose: print('Retrieving relevant chunks...')
    relevant = retriever.get_relevant_chunks(query, top_k=top_k)
    if verbose:
        for r in relevant:
            print(f" - {r['chunk_id']} (page {r['page']}) score={r['score']:.4f}")

    if use_openai:
        if verbose: print('Running OpenAI extraction...')
        try:
            results = extract_specs_from_chunks_openai(query, relevant, openai_model=openai_model)
        except Exception as e:
            print('OpenAI extraction failed:', e)
            results = relevant
    else:
        results = relevant

    save_json(results, out)
    if out_csv:
        save_csv(results, out_csv)

    if verbose: print('Saved JSON to', out)
    if out_csv and verbose: print('Saved CSV to', out_csv)

    return results


In [ ]:

# === Example run ===
# Adjust the following variables to run the pipeline in this notebook.
pdf_path = 'manual.pdf'   # <<== replace with your PDF path in the notebook environment
query = 'torque for brake caliper bolts'
use_openai = False        # set True to use OpenAI (you'll be prompted for key if not set)
top_k = 6
out_json = 'extracted_specs.json'

# Run (uncomment to execute)
#results = run_pipeline(pdf_path, query, use_openai=use_openai, top_k=top_k, out=out_json, verbose=True)
#results[:5] if isinstance(results, list) else results


In [ ]:

# Optional: write the single-file script to disk (vehicle_spec_extraction.py)
py_code = '''#!/usr/bin/env python3
# (This file is generated from the notebook. Edit as needed.)
# ... (trimmed for brevity) ...
'''
# To save the full .py, you can assemble the code blocks or paste your full script here.
# with open('vehicle_spec_extraction.py', 'w', encoding='utf-8') as f:
#     f.write(py_code)
print('If you want a .py file, paste your script into the cell and write it to disk as shown in the commented code.')
